In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configurer l'atténuation des erreurs

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** La version bêta d'un nouveau modèle d'exécution est désormais disponible. Le modèle d'exécution dirigée offre plus de flexibilité pour personnaliser ton flux de travail d'atténuation des erreurs. Consulte le guide [Modèle d'exécution dirigée](/guides/directed-execution-model) pour en savoir plus.


<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Les techniques d'atténuation des erreurs permettent aux utilisateurs de réduire les erreurs de circuit en
modélisant le bruit du dispositif au moment de l'exécution. Cela entraîne généralement
un surcoût de pré-traitement quantique lié à l'entraînement du modèle et
un surcoût de post-traitement classique pour atténuer les erreurs dans les résultats bruts
à l'aide du modèle généré.

La primitive Estimator prend en charge plusieurs techniques d'atténuation des erreurs, notamment [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec) et [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Consulte [Techniques d'atténuation et de suppression des erreurs](error-mitigation-and-suppression-techniques) pour une explication de chacune d'elles. Lors de l'utilisation des primitives, tu peux activer ou désactiver chaque méthode individuellement. Voir la section [Paramètres d'erreur personnalisés](#advanced-error) pour plus de détails.

> **Note:** Sampler ne prend pas en charge l'atténuation des erreurs, mais tu peux utiliser le package [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (atténuation des erreurs de mesure sans matrice) pour effectuer l'atténuation des erreurs localement.

Estimator prend également en charge le `resilience_level`. Le niveau de résilience spécifie le degré de résistance aux
erreurs. Des niveaux plus élevés produisent des résultats plus précis, au prix de
temps de traitement plus longs. Les niveaux de résilience peuvent être utilisés pour configurer
le compromis coût/précision lors de l'application de l'atténuation des erreurs à ta
requête de primitive. L'atténuation des erreurs réduit les erreurs (biais) dans les résultats en traitant
les sorties d'une collection, ou ensemble, de circuits connexes. Le
degré de réduction des erreurs dépend de la méthode appliquée. Le niveau de résilience
abstrait le choix détaillé de la méthode d'atténuation des erreurs pour permettre
aux utilisateurs de raisonner sur le compromis coût/précision adapté à
leur application.

Ainsi, chaque niveau correspond à une méthode ou à des méthodes avec
un niveau croissant de surcharge d'échantillonnage quantique pour te permettre d'expérimenter
différents compromis temps-précision. Le tableau suivant indique
quels niveaux et méthodes correspondantes sont disponibles pour chacune des
primitives.

> **Info:** L'atténuation des erreurs est spécifique à la tâche, donc les techniques que tu peux
> appliquer varient selon que tu échantillonnes une distribution ou que tu génères
> des valeurs d'espérance.

<span id="resilience-table"></span>

Estimator prend en charge les niveaux de résilience suivants. Sampler ne prend pas en charge les niveaux de résilience.

| Niveau de résilience | Définition                                                                                                      | Technique                                                                   |
|----------------------|-----------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------|
| 0                    | Aucune atténuation                                                                                              | Aucune                                                                      |
| 1 [Par défaut]       | Coûts d'atténuation minimaux : atténuation des erreurs liées aux erreurs de lecture                            | Twirled Readout Error eXtinction (TREX) avec twirling de mesure             |
| 2                    | Coûts d'atténuation moyens. Réduit généralement le biais dans les estimateurs, sans garantie de résultat sans biais. | Niveau 1 + Zero Noise Extrapolation (ZNE) et twirling de portes

> **Info:** Les niveaux de résilience sont actuellement en version bêta, donc la surcharge d'échantillonnage et
> la qualité des solutions varieront d'un circuit à l'autre. De nouvelles fonctionnalités,
> des options avancées et des outils de gestion seront publiés de façon progressive.
> Les méthodes spécifiques d'atténuation des erreurs ne sont pas garanties d'être
> appliquées à chaque niveau de résilience.

## Configurer Estimator avec les niveaux de résilience
Tu peux utiliser les niveaux de résilience pour spécifier des techniques d'atténuation des erreurs, ou définir des techniques personnalisées individuellement comme décrit dans [Paramètres d'erreur personnalisés.](#advanced-error)

<details>
<summary>Niveau de résilience 0</summary>

Aucune atténuation des erreurs n'est appliquée au programme utilisateur.

</details>

<details>
<summary>Niveau de résilience 1</summary>

Le niveau 1 applique l'**atténuation des erreurs de lecture** et le **twirling de mesure** en utilisant une technique sans modèle connue
sous le nom de Twirled Readout Error eXtinction (TREX). Elle réduit les erreurs de mesure
en diagonalisant le canal de bruit associé à la mesure en
faisant basculer aléatoirement les qubits via des portes X juste avant la mesure. Un
terme de remise à l'échelle du canal de bruit diagonal est appris par
l'étalonnage de circuits aléatoires initialisés dans l'état zéro. Cela permet
au service de supprimer le biais des valeurs d'espérance résultant du
bruit de lecture. Cette approche est décrite plus en détail dans [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Niveau de résilience 2</summary>

Le niveau 2 applique les **techniques d'atténuation des erreurs incluses dans le niveau 1** et applique également le **twirling de portes** et utilise la **méthode Zero Noise Extrapolation (ZNE)**. ZNE calcule une
valeur d'espérance de l'observable pour différents facteurs de bruit
(étape d'amplification), puis utilise les valeurs d'espérance mesurées pour
inférer la valeur d'espérance idéale à la limite de bruit zéro (étape
d'extrapolation). Cette approche tend à réduire les erreurs dans les valeurs d'espérance, mais
ne garantit pas un résultat sans biais.

![Cette image montre un graphe. L'axe des abscisses est intitulé Facteur d'amplification du bruit. L'axe des ordonnées est intitulé Valeur d'espérance. Une droite montante est intitulée Valeur atténuée. Les points proches de la droite sont des valeurs amplifiées par le bruit. Il y a une droite horizontale juste au-dessus de l'axe des abscisses intitulée Valeur exacte.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Illustration de la méthode ZNE")

Le surcoût de cette méthode évolue avec le nombre de facteurs de bruit. Les
paramètres par défaut échantillonnent la valeur d'espérance à trois facteurs de bruit,
ce qui entraîne un surcoût d'environ 3x lors de l'utilisation de ce niveau de résilience.

Dans le niveau 2, la méthode TREX fait basculer aléatoirement les qubits via des portes X juste avant la mesure,
et retourne le bit mesuré correspondant si une porte X a été appliquée. Cette approche est décrite plus en détail dans [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Exemple
L'interface `EstimatorV2` permet aux utilisateurs de travailler facilement avec la variété de
méthodes d'atténuation des erreurs pour réduire les erreurs dans les valeurs d'espérance des
observables. Le code suivant utilise Zero Noise Extrapolation et l'atténuation des erreurs de lecture en définissant simplement
`resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Paramètres d'erreur personnalisés
Tu peux activer et désactiver individuellement les méthodes d'atténuation et de suppression des erreurs, notamment le découplage dynamique, le twirling de portes et de mesures, l'atténuation des erreurs de mesure, PEC et ZNE. Consulte [Techniques d'atténuation et de suppression des erreurs](error-mitigation-and-suppression-techniques) pour une explication de chacune d'elles.

> **Note:** - Toutes les options ne sont pas disponibles pour les deux primitives. Consulte le [tableau des options disponibles](runtime-options-overview#options-table) pour la liste des options disponibles.
> - Toutes les méthodes ne fonctionnent pas ensemble sur tous les types de circuits. Consulte le [tableau de compatibilité des fonctionnalités](runtime-options-overview#options-compatibility-table) pour plus de détails.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">